# Unified multimodal movie dataset creation

This notebook replaces the old two-step flow:

1. `merge.ipynb` created an initial dataset.
2. `build_multimodal_dataset.py` repaired that dataset with `TMDB_all_movies.csv`.

The workflow below builds the enriched dataset directly in one pass: load raw
sources, normalize IDs, merge movie-level metadata once, apply TMDB fallback
immediately, then write model-ready rating files.


## Step 0 - Configuration

Run this notebook from either the repository root or the `dataset/` directory.
The notebook auto-detects the data directory and writes the final dataset to
`MultimodalMovieDataset_v2`.


In [1]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

# The notebook can be launched from the repository root or from dataset/.
DATASET_DIR = Path.cwd()
if not (DATASET_DIR / "ML25M").exists() and (DATASET_DIR / "dataset" / "ML25M").exists():
    DATASET_DIR = DATASET_DIR / "dataset"

OUTPUT_DIR = DATASET_DIR / "MultimodalMovieDataset_v2"
TMDB_IMAGE_BASE = "https://image.tmdb.org/t/p/w342"

SAMPLE_SIZE = 500_000
SEED = 42
CHUNKSIZE = 1_000_000
WRITE_FULL_MODEL_DF = False  # Set True only when you really need the huge full denormalized file.

PATHS = {
    "ml_movies": DATASET_DIR / "ML25M" / "movies.csv",
    "ml_links": DATASET_DIR / "ML25M" / "links.csv",
    "ml_ratings": DATASET_DIR / "ML25M" / "ratings.csv",
    "poster_csv": DATASET_DIR / "MovieGenreFromItsPoster" / "MovieGenre.csv",
    "movies_metadata": DATASET_DIR / "TheMoviesDataset" / "movies_metadata.csv",
    "tmdb_all_movies": DATASET_DIR / "TMDB" / "TMDB_all_movies.csv",
}

print(f"DATASET_DIR: {DATASET_DIR.resolve()}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR.resolve()}")


DATASET_DIR: C:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset
OUTPUT_DIR:  C:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset\MultimodalMovieDataset_v2


## Step 1 - Source policy

Used sources:

- MovieLens `movies.csv`, `links.csv`, and `ratings.csv` provide movie IDs,
  IMDb/TMDB mappings, and collaborative ratings.
- `MovieGenreFromItsPoster/MovieGenre.csv` provides legacy poster URLs, IMDb
  scores, and poster-derived genres.
- `TheMoviesDataset/movies_metadata.csv` is used when present for TMDB-style
  metadata.
- `TMDB/TMDB_all_movies.csv` is always used as fallback metadata and as the
  preferred poster source.

Not used in the final dataset: MovieLens tags/genome files, TheMoviesDataset
credits, and keywords. Those files are useful for other experiments, but the
current recommender does not consume them, so this notebook does not load large
unused tables.


In [2]:
source_report = pd.DataFrame(
    [
        {"source": name, "path": str(path), "exists": path.exists(), "required": name != "movies_metadata"}
        for name, path in PATHS.items()
    ]
)
display(source_report)

missing_required = source_report.query("required and not exists")
if not missing_required.empty:
    raise FileNotFoundError("Missing required source files:\n" + missing_required[["source", "path"]].to_string(index=False))

if not PATHS["movies_metadata"].exists():
    print("Optional TheMoviesDataset/movies_metadata.csv was not found; TMDB_all_movies.csv will fill metadata directly.")


,source,path,exists,required
0,ml_movies,c:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset\ML25M\movies.csv,True,True
1,ml_links,c:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset\ML25M\links.csv,True,True
2,ml_ratings,c:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset\ML25M\ratings.csv,True,True
3,poster_csv,c:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset\MovieGenreFromItsPoster\MovieGenre.csv,True,True
4,movies_metadata,c:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset\TheMoviesDataset\movies_metadata.csv,False,False
5,tmdb_all_movies,c:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset\TMDB\TMDB_all_movies.csv,True,True


Optional TheMoviesDataset/movies_metadata.csv was not found; TMDB_all_movies.csv will fill metadata directly.


## Step 2 - Shared helpers

All normalization, genre parsing, poster URL handling, sampling, and output
writing lives here once. Later cells call these helpers instead of repeating
merge and repair code.


In [ ]:
def clean_string(value):
    if value is None:
        return np.nan
    try:
        if pd.isna(value):
            return np.nan
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    return text if text and text.lower() not in {"nan", "none", "<na>"} else np.nan


def is_empty_value(value):
    if isinstance(value, list):
        return len(value) == 0
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        return False
    return str(value).strip() == ""


def normalize_tmdb_series(series):
    out = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    return out.mask(out.isin(["", "nan", "None", "<NA>"]))


def normalize_imdb_series(series):
    out = (
        series.astype("string")
        .str.strip()
        .str.replace(r"(?i)^tt", "", regex=True)
        .str.replace(r"\.0$", "", regex=True)
        .str.lstrip("0")
    )
    return out.mask(out.isin(["", "nan", "None", "<NA>"]))


def split_pipe_genres(value):
    value = clean_string(value)
    if pd.isna(value) or value == "(no genres listed)":
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]


def parse_tmdb_metadata_genres(value):
    value = clean_string(value)
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return []
    if not isinstance(parsed, list):
        return []
    return [str(item.get("name", "")).strip() for item in parsed if isinstance(item, dict) and str(item.get("name", "")).strip()]


def parse_comma_genres(value):
    value = clean_string(value)
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",") if part.strip()]


def standardize_genres(genres):
    genre_map = {
        "Musical": "Music",
        "Sci-Fi": "Science Fiction",
        "Children": "Family",
        "Film-Noir": "Noir",
    }
    out = []
    for genre in genres if isinstance(genres, list) else []:
        clean = genre_map.get(str(genre).strip(), str(genre).strip())
        if clean and clean not in out:
            out.append(clean)
    return out


def ordered_union(*genre_lists):
    out = []
    for genre_list in genre_lists:
        for genre in standardize_genres(genre_list):
            if genre not in out:
                out.append(genre)
    return out


def first_non_null(*values):
    for value in values:
        if not is_empty_value(value):
            return value
    return np.nan


def poster_url_from_path(path):
    path = clean_string(path)
    if pd.isna(path):
        return np.nan
    text = str(path)
    if text.startswith(("http://", "https://")):
        return text
    if not text.startswith("/"):
        text = "/" + text
    return f"{TMDB_IMAGE_BASE}{text}"


def extract_year_from_title(title):
    year = pd.Series(title, dtype="string").str.extract(r"\((\d{4})\)\s*$", expand=False)
    return pd.to_numeric(year, errors="coerce")


def score_metadata_rows(df, columns):
    score = pd.Series(0, index=df.index, dtype="int64")
    weights = {
        "poster": 16,
        "overview": 8,
        "genres": 4,
        "vote_count": 2,
        "title": 1,
    }
    for label, column in columns.items():
        if column in df.columns:
            score += df[column].notna().astype("int64") * weights[label]
    return score


def best_by_key(df, key):
    if key not in df.columns:
        return df.iloc[0:0].copy()
    ranked = df.dropna(subset=[key]).sort_values("source_quality_score", ascending=False)
    return ranked.drop_duplicates(key, keep="first")


def fill_column_from_join(df, target, fallback):
    if target not in df.columns:
        df[target] = np.nan
    if fallback not in df.columns:
        return
    mask = df[target].apply(is_empty_value)
    df.loc[mask, target] = df.loc[mask, fallback]


def write_dataset_readme(output_dir, audit, write_full_model_df):
    lines = [
        "# Multimodal Movie Dataset v2",
        "",
        "Built by `dataset/merge.ipynb`, the unified dataset creation notebook.",
        "",
        "Construction summary:",
        "- MovieLens provides collaborative ratings and stable movie IDs.",
        "- MovieLens links normalize IMDb and TMDB IDs for cross-source joins.",
        "- MovieGenreFromItsPoster contributes legacy poster URLs, IMDb scores, and poster genres.",
        "- TheMoviesDataset metadata is used when available.",
        "- TMDB_all_movies is applied in the same build as fallback metadata and as the preferred poster source.",
        "",
        "Files:",
        "- `final_movie_features_clean.csv`: one row per movie.",
        "- `final_ratings_clean.csv`: all MovieLens ratings with user/movie indices.",
        "- `model_sample_500k.csv`: sampled ratings joined with movie features.",
        "- `movie2idx.csv`, `user2idx.csv`: index mappings.",
        "- `audit_report.csv`: coverage and row-count checks.",
        "- `poster_embeddings.npy`: created later by `extract_poster_embeddings.py`, not by this notebook.",
    ]
    if write_full_model_df:
        lines.append("- `model_df_clean.csv`: full ratings joined with movie features.")
    else:
        lines.append("- `model_df_clean.csv`: skipped by default because it is very large.")
    lines.extend([
        "",
        "Next step:",
        "- Run `python extract_poster_embeddings.py --data-dir dataset/MultimodalMovieDataset_v2` from the project root to add visual poster features.",
        "",
        "Audit:",
    ])
    lines.extend([f"- {key}: {value}" for key, value in audit.items()])
    (output_dir / "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


## Step 3 - Load and prepare movie-level sources

This step builds one movie-level table from MovieLens IDs first. Poster
metadata, optional TheMoviesDataset metadata, and TMDB fallback data are all
deduplicated before joining, so every later merge stays one row per MovieLens
movie.


In [4]:
def load_poster_csv(path):
    for encoding in ("utf-8", "latin1", "ISO-8859-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not load poster CSV with known encodings: {path}")


def prepare_movielens_movies():
    ml_movies = pd.read_csv(PATHS["ml_movies"], low_memory=False)
    ml_links = pd.read_csv(PATHS["ml_links"], low_memory=False)

    ml_links = ml_links.copy()
    ml_links["imdb_id_clean"] = normalize_imdb_series(ml_links["imdbId"])
    ml_links["tmdb_id_clean"] = normalize_tmdb_series(ml_links["tmdbId"])

    movies = ml_movies.merge(
        ml_links[["movieId", "imdb_id_clean", "tmdb_id_clean"]],
        on="movieId",
        how="left",
    )
    movies["ml_genres_list"] = movies["genres"].apply(split_pipe_genres).apply(standardize_genres)
    movies["ml_title_year"] = extract_year_from_title(movies["title"])
    return movies


def prepare_poster_metadata():
    poster = load_poster_csv(PATHS["poster_csv"])
    poster = poster.rename(
        columns={
            "Title": "poster_title",
            "Genre": "poster_genres",
            "Poster": "poster_url_original",
            "IMDB Score": "poster_imdb_score",
        }
    )
    poster["imdb_id_clean"] = normalize_imdb_series(poster["imdbId"])
    poster["poster_genres_list"] = poster["poster_genres"].apply(split_pipe_genres).apply(standardize_genres)
    poster["poster_imdb_score"] = pd.to_numeric(poster["poster_imdb_score"], errors="coerce")
    poster["source_quality_score"] = score_metadata_rows(
        poster,
        {
            "poster": "poster_url_original",
            "overview": "poster_title",
            "genres": "poster_genres",
            "vote_count": "poster_imdb_score",
            "title": "poster_title",
        },
    )
    keep = [
        "imdb_id_clean",
        "poster_title",
        "poster_imdb_score",
        "poster_genres_list",
        "poster_url_original",
        "source_quality_score",
    ]
    return best_by_key(poster[keep], "imdb_id_clean").drop(columns="source_quality_score")


movies_base = prepare_movielens_movies()
poster_by_imdb = prepare_poster_metadata()

print("MovieLens movies:", movies_base.shape)
print("Poster rows after IMDb de-duplication:", poster_by_imdb.shape)
display(movies_base.head(3))
display(poster_by_imdb.head(3))


MovieLens movies: (62423, 7)
Poster rows after IMDb de-duplication: (39515, 5)


,movieId,title,genres,imdb_id_clean,tmdb_id_clean,ml_genres_list,ml_title_year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,"[Adventure, Animation, Family, Comedy, Fantasy]",1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,"[Adventure, Family, Fantasy]",1995
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,"[Comedy, Romance]",1995


,imdb_id_clean,poster_title,poster_imdb_score,poster_genres_list,poster_url_original
40106,79142,Xiao zi ming da (1979),6.5,"[Action, Comedy]","https://images-na.ssl-images-amazon.com/images/M/MV5BMjA0NTQ2MTc0OF5BMl5BanBnXkFtZTYwMzU0NjQ5._V1_UX182_CR0,0,182,268_AL_.jpg"
0,114709,Toy Story (1995),8.3,"[Animation, Adventure, Comedy]","https://images-na.ssl-images-amazon.com/images/M/MV5BMDU2ZWJlMjktMTRhMy00ZTA5LWEzNDgtYmNmZTEwZTViZWJkXkEyXkFqcGdeQXVyNDQ2OTk4MzI@._V1_UX182_CR0,0,182,268_AL_.jpg"
1,113497,Jumanji (1995),6.9,"[Action, Adventure, Family]","https://images-na.ssl-images-amazon.com/images/M/MV5BZTk2ZmUwYmEtNTcwZS00YmMyLWFkYjMtNTRmZDA3YWExMjc2XkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UY268_CR10,0,182,268_AL_.jpg"


## Step 4 - Prepare optional TheMoviesDataset metadata

If `movies_metadata.csv` is present, it is normalized and used first for
TMDB-style metadata. If it is missing, the notebook creates empty metadata
tables and continues with TMDB fallback only.


In [5]:
def prepare_movies_metadata(path):
    meta_columns = [
        "id",
        "imdb_id",
        "title",
        "original_title",
        "genres",
        "poster_path",
        "vote_average",
        "vote_count",
        "popularity",
        "runtime",
        "release_date",
        "overview",
        "original_language",
    ]
    output_columns = [
        "tmdb_id_clean",
        "imdb_id_clean",
        "meta_title",
        "meta_original_title",
        "meta_genres_list",
        "meta_poster_path",
        "meta_vote_average",
        "meta_vote_count",
        "meta_popularity",
        "meta_runtime",
        "meta_release_year",
        "meta_overview",
        "meta_original_language",
        "source_quality_score",
    ]

    if not path.exists():
        empty = pd.DataFrame(columns=output_columns)
        return empty.copy(), empty.copy()

    meta = pd.read_csv(path, usecols=lambda col: col in meta_columns, low_memory=False)
    meta["tmdb_id_clean"] = normalize_tmdb_series(meta["id"])
    meta["imdb_id_clean"] = normalize_imdb_series(meta["imdb_id"])
    meta = meta.rename(
        columns={
            "title": "meta_title",
            "original_title": "meta_original_title",
            "genres": "meta_genres_raw",
            "poster_path": "meta_poster_path",
            "vote_average": "meta_vote_average",
            "vote_count": "meta_vote_count",
            "popularity": "meta_popularity",
            "runtime": "meta_runtime",
            "release_date": "meta_release_date",
            "overview": "meta_overview",
            "original_language": "meta_original_language",
        }
    )
    meta["meta_genres_list"] = meta["meta_genres_raw"].apply(parse_tmdb_metadata_genres).apply(standardize_genres)
    meta["meta_release_year"] = pd.to_datetime(meta["meta_release_date"], errors="coerce").dt.year
    for column in ["meta_vote_average", "meta_vote_count", "meta_popularity", "meta_runtime"]:
        meta[column] = pd.to_numeric(meta[column], errors="coerce")
    meta["source_quality_score"] = score_metadata_rows(
        meta,
        {
            "poster": "meta_poster_path",
            "overview": "meta_overview",
            "genres": "meta_genres_raw",
            "vote_count": "meta_vote_count",
            "title": "meta_title",
        },
    )
    meta = meta[output_columns]
    return best_by_key(meta, "tmdb_id_clean"), best_by_key(meta, "imdb_id_clean")


meta_by_tmdb, meta_by_imdb = prepare_movies_metadata(PATHS["movies_metadata"])
print("TheMoviesDataset by TMDB:", meta_by_tmdb.shape)
print("TheMoviesDataset by IMDb:", meta_by_imdb.shape)
display(meta_by_tmdb.head(3))


TheMoviesDataset by TMDB: (0, 14)
TheMoviesDataset by IMDb: (0, 14)


,tmdb_id_clean,imdb_id_clean,meta_title,meta_original_title,meta_genres_list,meta_poster_path,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,meta_release_year,meta_overview,meta_original_language,source_quality_score


## Step 5 - Prepare TMDB fallback metadata

TMDB fallback rows are scored by useful metadata coverage. The best row per TMDB
ID and the best row per IMDb ID are kept, so a movie can be recovered even when
one identifier is missing or inconsistent.


In [6]:
def prepare_tmdb_all(path):
    usecols = [
        "id",
        "title",
        "vote_average",
        "vote_count",
        "release_date",
        "runtime",
        "imdb_id",
        "original_language",
        "original_title",
        "overview",
        "popularity",
        "genres",
        "poster_path",
    ]
    tmdb = pd.read_csv(path, usecols=usecols, low_memory=False)
    tmdb["tmdb_id_clean"] = normalize_tmdb_series(tmdb["id"])
    tmdb["imdb_id_clean"] = normalize_imdb_series(tmdb["imdb_id"])
    tmdb = tmdb.rename(
        columns={
            "title": "tmdb_all_title",
            "original_title": "tmdb_all_original_title",
            "genres": "tmdb_all_genres_raw",
            "poster_path": "tmdb_all_poster_path",
            "vote_average": "tmdb_all_vote_average",
            "vote_count": "tmdb_all_vote_count",
            "popularity": "tmdb_all_popularity",
            "runtime": "tmdb_all_runtime",
            "release_date": "tmdb_all_release_date",
            "overview": "tmdb_all_overview",
            "original_language": "tmdb_all_original_language",
        }
    )
    tmdb["tmdb_all_genres_list"] = tmdb["tmdb_all_genres_raw"].apply(parse_comma_genres).apply(standardize_genres)
    tmdb["tmdb_all_release_year"] = pd.to_datetime(tmdb["tmdb_all_release_date"], errors="coerce").dt.year
    tmdb["tmdb_all_poster_url"] = tmdb["tmdb_all_poster_path"].apply(poster_url_from_path)
    for column in ["tmdb_all_vote_average", "tmdb_all_vote_count", "tmdb_all_popularity", "tmdb_all_runtime"]:
        tmdb[column] = pd.to_numeric(tmdb[column], errors="coerce")
    tmdb["source_quality_score"] = score_metadata_rows(
        tmdb,
        {
            "poster": "tmdb_all_poster_path",
            "overview": "tmdb_all_overview",
            "genres": "tmdb_all_genres_raw",
            "vote_count": "tmdb_all_vote_count",
            "title": "tmdb_all_title",
        },
    )
    keep = [
        "tmdb_id_clean",
        "imdb_id_clean",
        "tmdb_all_title",
        "tmdb_all_original_title",
        "tmdb_all_genres_list",
        "tmdb_all_poster_path",
        "tmdb_all_poster_url",
        "tmdb_all_vote_average",
        "tmdb_all_vote_count",
        "tmdb_all_popularity",
        "tmdb_all_runtime",
        "tmdb_all_release_year",
        "tmdb_all_overview",
        "tmdb_all_original_language",
        "source_quality_score",
    ]
    tmdb = tmdb[keep]
    return best_by_key(tmdb, "tmdb_id_clean"), best_by_key(tmdb, "imdb_id_clean")


tmdb_by_tmdb, tmdb_by_imdb = prepare_tmdb_all(PATHS["tmdb_all_movies"])
print("TMDB fallback by TMDB:", tmdb_by_tmdb.shape)
print("TMDB fallback by IMDb:", tmdb_by_imdb.shape)
display(tmdb_by_tmdb.head(3))


TMDB fallback by TMDB: (1178829, 15)
TMDB fallback by IMDb: (660920, 15)


,tmdb_id_clean,imdb_id_clean,tmdb_all_title,tmdb_all_original_title,tmdb_all_genres_list,tmdb_all_poster_path,tmdb_all_poster_url,tmdb_all_vote_average,tmdb_all_vote_count,tmdb_all_popularity,tmdb_all_runtime,tmdb_all_release_year,tmdb_all_overview,tmdb_all_original_language,source_quality_score
1178756,1664496,<NA>,Elephants Can Remember,Elephants Can Remember,"[Crime, Drama, Mystery, Thriller, TV Movie]",/daLiT8fR6whwedxZ10JifoaKIc4.jpg,https://image.tmdb.org/t/p/w342/daLiT8fR6whwedxZ10JifoaKIc4.jpg,0.0,0.0,0.0000,89.0,2014.0,Ariadne Oliver becomes an amateur sleuth when her goddaughter tasks her to find out the truth behind her parents' mysterious deaths.,en,31
1178755,1664493,<NA>,Not for the Wide World,Not for the Wide World,"[War, Romance]",/6mieXUy0Edy9Wu4zqFTif0nJO2h.jpg,https://image.tmdb.org/t/p/w342/6mieXUy0Edy9Wu4zqFTif0nJO2h.jpg,10.0,1.0,0.0000,10.0,2026.0,Two soliders from messina set off for war. Will there love survive the gunfire?,en,31
31,64,287467,Talk to Her,Hable con ella,"[Drama, Romance]",/fWDbQlOWOqjR5jZm98KjGyYmUOw.jpg,https://image.tmdb.org/t/p/w342/fWDbQlOWOqjR5jZm98KjGyYmUOw.jpg,7.6,1417.0,1.8774,112.0,2002.0,Two men share an odd friendship while they care for two women who are both in deep comas.,es,31


## Step 6 - Merge movie features once

Priority rules:

- Title: TheMoviesDataset title, then TMDB fallback title, then poster title,
  then MovieLens title.
- Poster URL: TMDB fallback poster, then TheMoviesDataset poster path, then
  legacy poster CSV URL.
- Genres: ordered union of TheMoviesDataset genres, TMDB fallback genres,
  poster genres, and MovieLens genres.
- Numeric metadata: TheMoviesDataset values first, then TMDB fallback values,
  with medians used only for numeric fields still missing after fallback.


In [7]:
def attach_source_by_two_keys(movies, by_tmdb, by_imdb, source_columns):
    if "tmdb_id_clean" not in movies.columns and "imdb_id_clean" not in movies.columns:
        raise KeyError("Movies must contain at least one source identifier column: tmdb_id_clean or imdb_id_clean.")

    tmdb_join = by_tmdb.drop(columns=["source_quality_score", "imdb_id_clean"], errors="ignore")
    out = movies.merge(
        tmdb_join,
        on="tmdb_id_clean",
        how="left",
    )

    imdb_join = by_imdb.drop(columns=["source_quality_score", "tmdb_id_clean"], errors="ignore").copy()
    if "imdb_id_clean" not in imdb_join.columns:
        raise KeyError("by_imdb must contain imdb_id_clean before join.")
    imdb_join = imdb_join.rename(columns={"imdb_id_clean": "imdb_id_clean_imdb_join"})
    imdb_join = imdb_join.rename(
        columns={col: f"{col}_imdb_join" for col in imdb_join.columns if col != "imdb_id_clean_imdb_join"}
    )

    out = out.merge(
        imdb_join,
        left_on="imdb_id_clean",
        right_on="imdb_id_clean_imdb_join",
        how="left",
    )
    for column in source_columns:
        fill_column_from_join(out, column, f"{column}_imdb_join")
    drop_cols = [column for column in out.columns if column.endswith("_imdb_join")]
    return out.drop(columns=drop_cols)


meta_columns = [
    "meta_title",
    "meta_original_title",
    "meta_genres_list",
    "meta_poster_path",
    "meta_vote_average",
    "meta_vote_count",
    "meta_popularity",
    "meta_runtime",
    "meta_release_year",
    "meta_overview",
    "meta_original_language",
]

tmdb_columns = [
    "tmdb_all_title",
    "tmdb_all_original_title",
    "tmdb_all_genres_list",
    "tmdb_all_poster_path",
    "tmdb_all_poster_url",
    "tmdb_all_vote_average",
    "tmdb_all_vote_count",
    "tmdb_all_popularity",
    "tmdb_all_runtime",
    "tmdb_all_release_year",
    "tmdb_all_overview",
    "tmdb_all_original_language",
]

movies = movies_base.merge(poster_by_imdb, on="imdb_id_clean", how="left")
movies = attach_source_by_two_keys(movies, meta_by_tmdb, meta_by_imdb, meta_columns)
movies = attach_source_by_two_keys(movies, tmdb_by_tmdb, tmdb_by_imdb, tmdb_columns)

for column in ["poster_genres_list", "meta_genres_list", "tmdb_all_genres_list"]:
    if column not in movies.columns:
        movies[column] = [[] for _ in range(len(movies))]
    movies[column] = movies[column].apply(lambda value: value if isinstance(value, list) else [])

movies["final_title"] = movies.apply(
    lambda row: first_non_null(row.get("meta_title"), row.get("tmdb_all_title"), row.get("poster_title"), row.get("title")),
    axis=1,
)
movies["original_title"] = movies.apply(
    lambda row: first_non_null(row.get("meta_original_title"), row.get("tmdb_all_original_title"), row.get("final_title")),
    axis=1,
)
movies["final_genres_list"] = movies.apply(
    lambda row: ordered_union(
        row.get("meta_genres_list"),
        row.get("tmdb_all_genres_list"),
        row.get("poster_genres_list"),
        row.get("ml_genres_list"),
    ),
    axis=1,
)
movies["final_genres_str"] = movies["final_genres_list"].apply("|".join)

movies["meta_overview"] = movies.apply(lambda row: first_non_null(row.get("meta_overview"), row.get("tmdb_all_overview")), axis=1)
movies["meta_original_language"] = movies.apply(
    lambda row: first_non_null(row.get("meta_original_language"), row.get("tmdb_all_original_language")),
    axis=1,
)

numeric_pairs = {
    "meta_vote_average": "tmdb_all_vote_average",
    "meta_vote_count": "tmdb_all_vote_count",
    "meta_popularity": "tmdb_all_popularity",
    "meta_runtime": "tmdb_all_runtime",
}
for target, fallback in numeric_pairs.items():
    movies[target] = pd.to_numeric(movies[target], errors="coerce")
    movies[fallback] = pd.to_numeric(movies[fallback], errors="coerce")
    movies[target] = movies[target].where(movies[target].notna(), movies[fallback])

movies["release_year"] = pd.to_numeric(movies.get("meta_release_year"), errors="coerce")
movies["release_year"] = movies["release_year"].where(movies["release_year"].notna(), pd.to_numeric(movies.get("tmdb_all_release_year"), errors="coerce"))
movies["release_year"] = movies["release_year"].where(movies["release_year"].notna(), movies["ml_title_year"])

meta_poster_url = movies["meta_poster_path"].apply(poster_url_from_path)
tmdb_poster_url = movies["tmdb_all_poster_url"].apply(clean_string)
legacy_poster_url = movies["poster_url_original"].apply(clean_string)

movies["poster_url"] = tmdb_poster_url.where(tmdb_poster_url.notna(), meta_poster_url)
movies["poster_url"] = movies["poster_url"].where(movies["poster_url"].notna(), legacy_poster_url)
movies["poster_url_source"] = np.select(
    [tmdb_poster_url.notna(), meta_poster_url.notna(), legacy_poster_url.notna()],
    ["tmdb_all", "movies_metadata", "moviegenre_imdb"],
    default="missing",
)
movies["meta_poster_path"] = movies["meta_poster_path"].where(movies["meta_poster_path"].notna(), movies["tmdb_all_poster_path"])

for column in ["poster_imdb_score", "meta_vote_average", "meta_vote_count", "meta_popularity", "meta_runtime"]:
    movies[column] = pd.to_numeric(movies[column], errors="coerce")
    median = movies[column].median()
    if pd.notna(median):
        movies[column] = movies[column].fillna(median)

movie_ids = sorted(movies["movieId"].dropna().astype(int).unique())
movie2idx = pd.DataFrame({"movieId": movie_ids, "movie_idx": range(len(movie_ids))})
movies = movies.merge(movie2idx, on="movieId", how="left")

all_genres = sorted({genre for genres in movies["final_genres_list"] for genre in genres})
genre2idx = {genre: idx for idx, genre in enumerate(all_genres)}


def to_multihot_string(genres):
    vector = np.zeros(len(genre2idx), dtype=np.int8)
    for genre in genres if isinstance(genres, list) else []:
        if genre in genre2idx:
            vector[genre2idx[genre]] = 1
    return "[" + " ".join(map(str, vector.tolist())) + "]"


movies["genre_multihot"] = movies["final_genres_list"].apply(to_multihot_string)

final_movie_columns = [
    "movieId",
    "imdb_id_clean",
    "tmdb_id_clean",
    "title",
    "final_title",
    "original_title",
    "ml_genres_list",
    "poster_genres_list",
    "meta_genres_list",
    "tmdb_all_genres_list",
    "final_genres_list",
    "final_genres_str",
    "poster_url",
    "poster_url_source",
    "poster_url_original",
    "tmdb_all_poster_url",
    "meta_poster_path",
    "poster_imdb_score",
    "meta_vote_average",
    "meta_vote_count",
    "meta_popularity",
    "meta_runtime",
    "release_year",
    "meta_original_language",
    "meta_overview",
    "movie_idx",
    "genre_multihot",
]
final_movie_df = movies[final_movie_columns].sort_values("movie_idx").reset_index(drop=True)

print("final_movie_df:", final_movie_df.shape)
print("Unique genres:", len(all_genres))
display(final_movie_df.head(5))


final_movie_df: (62423, 27)
Unique genres: 29


,movieId,imdb_id_clean,tmdb_id_clean,title,final_title,original_title,ml_genres_list,poster_genres_list,meta_genres_list,tmdb_all_genres_list,final_genres_list,final_genres_str,poster_url,poster_url_source,poster_url_original,tmdb_all_poster_url,meta_poster_path,poster_imdb_score,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,release_year,meta_original_language,meta_overview,movie_idx,genre_multihot
0,1,114709,862,Toy Story (1995),Toy Story,Toy Story,"[Adventure, Animation, Family, Comedy, Fantasy]","[Animation, Adventure, Comedy]",[],"[Family, Comedy, Animation, Adventure]","[Family, Comedy, Animation, Adventure, Fantasy]",Family|Comedy|Animation|Adventure|Fantasy,https://image.tmdb.org/t/p/w342/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,tmdb_all,"https://images-na.ssl-images-amazon.com/images/M/MV5BMDU2ZWJlMjktMTRhMy00ZTA5LWEzNDgtYmNmZTEwZTViZWJkXkEyXkFqcGdeQXVyNDQ2OTk4MzI@._V1_UX182_CR0,0,182,268_AL_.jpg",https://image.tmdb.org/t/p/w342/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,8.3,7.970,19699.0,20.2315,81.0,1995.0,en,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circum...",0,[0 0 1 1 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
1,2,113497,8844,Jumanji (1995),Jumanji,Jumanji,"[Adventure, Family, Fantasy]","[Action, Adventure, Family]",[],"[Adventure, Fantasy, Family]","[Adventure, Fantasy, Family, Action]",Adventure|Fantasy|Family|Action,https://image.tmdb.org/t/p/w342/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,tmdb_all,"https://images-na.ssl-images-amazon.com/images/M/MV5BZTk2ZmUwYmEtNTcwZS00YmMyLWFkYjMtNTRmZDA3YWExMjc2XkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UY268_CR10,0,182,268_AL_.jpg",https://image.tmdb.org/t/p/w342/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,6.9,7.244,11199.0,3.1851,104.0,1995.0,en,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- in...",1,[1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
2,3,113228,15602,Grumpier Old Men (1995),Grumpier Old Men,Grumpier Old Men,"[Comedy, Romance]","[Comedy, Romance]",[],"[Romance, Comedy]","[Romance, Comedy]",Romance|Comedy,https://image.tmdb.org/t/p/w342/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,tmdb_all,"https://images-na.ssl-images-amazon.com/images/M/MV5BMjQxM2YyNjMtZjUxYy00OGYyLTg0MmQtNGE2YzNjYmUyZTY1XkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",https://image.tmdb.org/t/p/w342/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,6.6,6.500,420.0,2.2316,101.0,1995.0,en,"A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a restaurant at the local bait shop, alarming t...",2,[0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
3,4,114885,31357,Waiting to Exhale (1995),Waiting to Exhale,Waiting to Exhale,"[Comedy, Drama, Romance]","[Comedy, Drama, Romance]",[],"[Comedy, Drama, Romance]","[Comedy, Drama, Romance]",Comedy|Drama|Romance,https://image.tmdb.org/t/p/w342/8MprEuTY3EwkF9nBBPCUyjRjvs5.jpg,tmdb_all,"https://images-na.ssl-images-amazon.com/images/M/MV5BMTczMTMyMTgyM15BMl5BanBnXkFtZTcwOTc4OTQyMQ@@._V1_UY268_CR4,0,182,268_AL_.jpg",https://image.tmdb.org/t/p/w342/8MprEuTY3EwkF9nBBPCUyjRjvs5.jpg,/8MprEuTY3EwkF9nBBPCUyjRjvs5.jpg,5.7,6.200,196.0,2.1975,127.0,1995.0,en,"Cheated on, mistreated and stepped on, the women are holding their breath, waiting for the elusive ""good man"" to break a string of less-than-stellar lovers. Friends and confidants Vannah, Bernie, ...",3,[0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
4,5,113041,11862,Father of the Bride Part II (1995),Father of the Bride Part II,Father of the Bride Part II,[Comedy],"[Comedy, Family, Romance]",[],"[Comedy, Family]","[Comedy, Family, Romanc

## Step 7 - Create user and movie index mappings

Movie indices come from the final movie table. User indices are scanned from
`ratings.csv` in chunks, so the notebook does not need to hold all 25M ratings
in memory just to create the mapping.


In [8]:
def scan_ratings_for_users_and_count(ratings_path):
    users = set()
    total_rows = 0
    for chunk in pd.read_csv(ratings_path, usecols=["userId"], chunksize=CHUNKSIZE, low_memory=False):
        total_rows += len(chunk)
        users.update(chunk["userId"].dropna().astype(int).unique().tolist())
    return sorted(users), total_rows


user_ids, rating_row_count = scan_ratings_for_users_and_count(PATHS["ml_ratings"])
user2idx = pd.DataFrame({"userId": user_ids, "user_idx": range(len(user_ids))})

print("Ratings rows:", f"{rating_row_count:,}")
print("Users:", len(user2idx))
print("Movies:", len(movie2idx))
display(user2idx.head())
display(movie2idx.head())


Ratings rows: 25,000,095
Users: 162541
Movies: 62423


,userId,user_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


,movieId,movie_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


## Step 8 - Write final ratings and sampled model data

This is the only step that streams through the full ratings table. It writes
`final_ratings_clean.csv` and collects an exact `model_sample_500k.csv` sample,
then joins that sample with movie features.


In [9]:
def write_ratings_and_collect_sample(ratings_path, output_path, user2idx, movie2idx, total_rows):
    user_index = dict(zip(user2idx["userId"], user2idx["user_idx"]))
    movie_index = dict(zip(movie2idx["movieId"], movie2idx["movie_idx"]))
    target = min(SAMPLE_SIZE, total_rows)
    sample_probability = 1.0 if target == total_rows else min(1.0, target / total_rows * 1.5)
    rng = np.random.default_rng(SEED)
    sample_parts = []
    first_chunk = True
    missing_movie_ids = set()

    for chunk in pd.read_csv(ratings_path, chunksize=CHUNKSIZE, low_memory=False):
        chunk = chunk[["userId", "movieId", "rating", "timestamp"]].copy()
        chunk["user_idx"] = chunk["userId"].map(user_index)
        chunk["movie_idx"] = chunk["movieId"].map(movie_index)

        if chunk["user_idx"].isna().any():
            raise ValueError("Found ratings with userId missing from user2idx mapping.")
        if chunk["movie_idx"].isna().any():
            missing_movie_ids.update(chunk.loc[chunk["movie_idx"].isna(), "movieId"].dropna().astype(int).unique().tolist())
            chunk = chunk.dropna(subset=["movie_idx"])

        chunk["user_idx"] = chunk["user_idx"].astype("int64")
        chunk["movie_idx"] = chunk["movie_idx"].astype("int64")
        chunk.to_csv(output_path, index=False, mode="w" if first_chunk else "a", header=first_chunk)
        first_chunk = False

        mask = rng.random(len(chunk)) < sample_probability
        if mask.any():
            sample_parts.append(chunk.loc[mask].copy())
        if sum(len(part) for part in sample_parts) > target * 3:
            sample_parts = [pd.concat(sample_parts, ignore_index=True).sample(n=target, random_state=SEED)]

    sampled = pd.concat(sample_parts, ignore_index=True)
    if len(sampled) < target:
        raise RuntimeError(f"Sampling produced only {len(sampled):,} rows; expected {target:,}.")
    sampled = sampled.sample(n=target, random_state=SEED).reset_index(drop=True)
    return sampled, missing_movie_ids


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final_movie_df.to_csv(OUTPUT_DIR / "final_movie_features_clean.csv", index=False)
user2idx.to_csv(OUTPUT_DIR / "user2idx.csv", index=False)
movie2idx.to_csv(OUTPUT_DIR / "movie2idx.csv", index=False)

ratings_sample, missing_movie_ids = write_ratings_and_collect_sample(
    PATHS["ml_ratings"],
    OUTPUT_DIR / "final_ratings_clean.csv",
    user2idx,
    movie2idx,
    rating_row_count,
)

if missing_movie_ids:
    print(f"Dropped ratings for {len(missing_movie_ids)} movieIds missing from movies.csv.")

feature_columns = [
    "movieId",
    "movie_idx",
    "final_title",
    "final_genres_str",
    "poster_url",
    "poster_url_source",
    "poster_imdb_score",
    "meta_vote_average",
    "meta_vote_count",
    "meta_popularity",
    "meta_runtime",
    "release_year",
    "meta_original_language",
    "meta_overview",
]
model_sample_df = ratings_sample.merge(final_movie_df[feature_columns], on=["movieId", "movie_idx"], how="left")
model_sample_df.to_csv(OUTPUT_DIR / "model_sample_500k.csv", index=False)

print("Saved movie features, mappings, full indexed ratings, and sample model file.")
print("model_sample_df:", model_sample_df.shape)
display(model_sample_df.head())


Saved movie features, mappings, full indexed ratings, and sample model file.
model_sample_df: (500000, 18)


,userId,movieId,rating,timestamp,user_idx,movie_idx,final_title,final_genres_str,poster_url,poster_url_source,poster_imdb_score,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,release_year,meta_original_language,meta_overview
0,125655,4330,4.0,1171158819,125654,4225,The Scarlet Empress,Drama|History|Romance,https://image.tmdb.org/t/p/w342/lCoyv38br7T4hFPJF0OnfAGuzbp.jpg,tmdb_all,7.7,6.900,141.0,1.1058,104.0,1934.0,en,"During the 18th century, German noblewoman Sophia Frederica, who would later become Catherine the Great, travels to Moscow to marry the dimwitted Grand Duke Peter, the heir to the Russian throne. ..."
1,161337,141,3.0,1084486451,161336,139,The Birdcage,Comedy,https://image.tmdb.org/t/p/w342/hU2XeckncHS61TWZKDtw1BrKmOO.jpg,tmdb_all,7.0,7.100,1333.0,4.5761,119.0,1996.0,en,A gay cabaret owner and his drag queen partner agree to put up a false heterosexual front so that their son can introduce them to his fiancée's conservative parents.
2,88683,508,4.0,842381981,88682,503,Philadelphia,Drama,https://image.tmdb.org/t/p/w342/tFe5Yoo5zT495okA49bq1vPPkiV.jpg,tmdb_all,7.7,7.714,4560.0,4.7766,126.0,1993.0,en,"Two competing lawyers join forces to sue a prestigious law firm for AIDS discrimination. As their unlikely friendship develops, their courage overcomes the prejudice and corruption of their powerf..."
3,92345,48516,4.0,1185158463,92344,11124,The Departed,Drama|Thriller|Crime,https://image.tmdb.org/t/p/w342/nT97ifVT2J1yMQmeq20Qblg61T.jpg,tmdb_all,8.5,8.160,16021.0,13.8708,151.0,2006.0,en,"To take down South Boston's Irish Mafia, the police send in one of their own to infiltrate the underworld, not realizing the syndicate has done likewise. While an undercover cop curries favor with..."
4,16537,6378,2.0,1569198745,16536,6259,The Italian Job,Action|Crime|Thriller,https://image.tmdb.org/t/p/w342/eSkjK4kctyrWpFhxl35GPvSs6tI.jpg,tmdb_all,7.0,6.786,5945.0,7.2626,110.0,2003.0,en,"Charlie Croker pulled off the crime of a lifetime. The one thing that he didn't plan on was being double-crossed. Along with a drop-dead gorgeous safecracker, Croker and his team take off to re-st..."


## Step 9 - Optional full denormalized model file

`model_df_clean.csv` is very large because it repeats movie metadata for every
rating. The training notebook can reconstruct it from `final_ratings_clean.csv`
and `final_movie_features_clean.csv`, so this notebook skips it by default.


In [10]:
def write_full_model_df(ratings_path, movies, output_path):
    feature_columns = [
        "movieId",
        "movie_idx",
        "final_title",
        "final_genres_str",
        "poster_url",
        "poster_url_source",
        "poster_imdb_score",
        "meta_vote_average",
        "meta_vote_count",
        "meta_popularity",
        "meta_runtime",
        "release_year",
        "meta_original_language",
        "meta_overview",
    ]
    features = movies[feature_columns]
    first_chunk = True
    for chunk in pd.read_csv(ratings_path, chunksize=CHUNKSIZE, low_memory=False):
        out = chunk.merge(features, on=["movieId", "movie_idx"], how="left")
        out.to_csv(output_path, index=False, mode="w" if first_chunk else "a", header=first_chunk)
        first_chunk = False


if WRITE_FULL_MODEL_DF:
    write_full_model_df(OUTPUT_DIR / "final_ratings_clean.csv", final_movie_df, OUTPUT_DIR / "model_df_clean.csv")
    print("Saved model_df_clean.csv")
else:
    model_df_path = OUTPUT_DIR / "model_df_clean.csv"
    if model_df_path.exists():
        print("Existing model_df_clean.csv was left untouched.")
    else:
        print("Skipped model_df_clean.csv. Set WRITE_FULL_MODEL_DF = True to create it.")


Skipped model_df_clean.csv. Set WRITE_FULL_MODEL_DF = True to create it.


## Step 10 - Audit coverage and write dataset README

The audit summarizes the exact coverage after the unified build, including how
many final posters came from TMDB fallback versus older sources.


In [11]:
sample_movies = final_movie_df[final_movie_df["movie_idx"].isin(model_sample_df["movie_idx"].unique())]

audit = {
    "movies": int(len(final_movie_df)),
    "ratings_rows": int(rating_row_count),
    "users": int(len(user2idx)),
    "poster_url_original": int(final_movie_df["poster_url_original"].notna().sum()),
    "tmdb_all_poster_url": int(final_movie_df["tmdb_all_poster_url"].notna().sum()),
    "poster_url_final": int(final_movie_df["poster_url"].notna().sum()),
    "overview": int(final_movie_df["meta_overview"].notna().sum()),
    "genres": int(final_movie_df["final_genres_str"].astype(str).str.len().gt(0).sum()),
    "poster_source_tmdb_all": int((final_movie_df["poster_url_source"] == "tmdb_all").sum()),
    "poster_source_movies_metadata": int((final_movie_df["poster_url_source"] == "movies_metadata").sum()),
    "poster_source_moviegenre_imdb": int((final_movie_df["poster_url_source"] == "moviegenre_imdb").sum()),
    "poster_source_missing": int((final_movie_df["poster_url_source"] == "missing").sum()),
    "sample_rows": int(len(model_sample_df)),
    "sample_unique_movies": int(model_sample_df["movie_idx"].nunique()),
    "sample_rows_with_poster_url": int(model_sample_df["poster_url"].notna().sum()),
    "sample_unique_movies_with_poster_url": int(sample_movies["poster_url"].notna().sum()),
}

audit_df = pd.Series(audit, name="value").to_frame()
audit_df.to_csv(OUTPUT_DIR / "audit_report.csv")
write_dataset_readme(OUTPUT_DIR, audit, WRITE_FULL_MODEL_DF)

display(audit_df)
print(f"Audit and README saved to {OUTPUT_DIR}")


,value
movies,62423
ratings_rows,25000095
users,162541
poster_url_original,36262
tmdb_all_poster_url,60960
poster_url_final,61572
overview,61673
genres,62126
poster_source_tmdb_all,60960
poster_source_movies_metadata,0


Audit and README saved to c:\Users\murad\Desktop\master\Neural networks\GroupProject\final\dataset\MultimodalMovieDataset_v2


## Step 11 - Quick saved-file check

This final check reads only headers and a few rows from each output. It verifies
that the files exist without loading the huge rating table back into memory.


In [12]:
created_files = [
    "final_movie_features_clean.csv",
    "final_ratings_clean.csv",
    "model_sample_500k.csv",
    "movie2idx.csv",
    "user2idx.csv",
    "audit_report.csv",
    "README.md",
]
if WRITE_FULL_MODEL_DF:
    created_files.append("model_df_clean.csv")

for file_name in created_files:
    path = OUTPUT_DIR / file_name
    print("=" * 100)
    print(path.name)
    if not path.exists():
        print("Missing")
        continue
    print(f"Size: {path.stat().st_size / (1024 ** 2):.2f} MB")
    if path.suffix == ".csv":
        head = pd.read_csv(path, nrows=3, low_memory=False)
        print("Columns:", head.columns.tolist())
        display(head)
    else:
        print(path.read_text(encoding="utf-8")[:500])


final_movie_features_clean.csv
Size: 49.83 MB
Columns: ['movieId', 'imdb_id_clean', 'tmdb_id_clean', 'title', 'final_title', 'original_title', 'ml_genres_list', 'poster_genres_list', 'meta_genres_list', 'tmdb_all_genres_list', 'final_genres_list', 'final_genres_str', 'poster_url', 'poster_url_source', 'poster_url_original', 'tmdb_all_poster_url', 'meta_poster_path', 'poster_imdb_score', 'meta_vote_average', 'meta_vote_count', 'meta_popularity', 'meta_runtime', 'release_year', 'meta_original_language', 'meta_overview', 'movie_idx', 'genre_multihot']


,movieId,imdb_id_clean,tmdb_id_clean,title,final_title,original_title,ml_genres_list,poster_genres_list,meta_genres_list,tmdb_all_genres_list,final_genres_list,final_genres_str,poster_url,poster_url_source,poster_url_original,tmdb_all_poster_url,meta_poster_path,poster_imdb_score,meta_vote_average,meta_vote_count,meta_popularity,meta_runtime,release_year,meta_original_language,meta_overview,movie_idx,genre_multihot
0,1,114709,862,Toy Story (1995),Toy Story,Toy Story,"['Adventure', 'Animation', 'Family', 'Comedy', 'Fantasy']","['Animation', 'Adventure', 'Comedy']",[],"['Family', 'Comedy', 'Animation', 'Adventure']","['Family', 'Comedy', 'Animation', 'Adventure', 'Fantasy']",Family|Comedy|Animation|Adventure|Fantasy,https://image.tmdb.org/t/p/w342/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,tmdb_all,"https://images-na.ssl-images-amazon.com/images/M/MV5BMDU2ZWJlMjktMTRhMy00ZTA5LWEzNDgtYmNmZTEwZTViZWJkXkEyXkFqcGdeQXVyNDQ2OTk4MzI@._V1_UX182_CR0,0,182,268_AL_.jpg",https://image.tmdb.org/t/p/w342/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,8.3,7.970,19699.0,20.2315,81.0,1995.0,en,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circum...",0,[0 0 1 1 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
1,2,113497,8844,Jumanji (1995),Jumanji,Jumanji,"['Adventure', 'Family', 'Fantasy']","['Action', 'Adventure', 'Family']",[],"['Adventure', 'Fantasy', 'Family']","['Adventure', 'Fantasy', 'Family', 'Action']",Adventure|Fantasy|Family|Action,https://image.tmdb.org/t/p/w342/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,tmdb_all,"https://images-na.ssl-images-amazon.com/images/M/MV5BZTk2ZmUwYmEtNTcwZS00YmMyLWFkYjMtNTRmZDA3YWExMjc2XkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UY268_CR10,0,182,268_AL_.jpg",https://image.tmdb.org/t/p/w342/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,6.9,7.244,11199.0,3.1851,104.0,1995.0,en,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- in...",1,[1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
2,3,113228,15602,Grumpier Old Men (1995),Grumpier Old Men,Grumpier Old Men,"['Comedy', 'Romance']","['Comedy', 'Romance']",[],"['Romance', 'Comedy']","['Romance', 'Comedy']",Romance|Comedy,https://image.tmdb.org/t/p/w342/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,tmdb_all,"https://images-na.ssl-images-amazon.com/images/M/MV5BMjQxM2YyNjMtZjUxYy00OGYyLTg0MmQtNGE2YzNjYmUyZTY1XkEyXkFqcGdeQXVyMTQxNzMzNDI@._V1_UX182_CR0,0,182,268_AL_.jpg",https://image.tmdb.org/t/p/w342/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,6.6,6.500,420.0,2.2316,101.0,1995.0,en,"A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a restaurant at the local bait shop, alarming t...",2,[0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]


final_ratings_clean.csv
Size: 915.67 MB
Columns: ['userId', 'movieId', 'rating', 'timestamp', 'user_idx', 'movie_idx']


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,296,5.0,1147880044,0,292
1,1,306,3.5,1147868817,0,302
2,1,307,5.0,1147868828,0,303


model_sample_500k.csv
Size: 229.46 MB
Columns: ['userId', 'movieId', 'rating', 'timestamp', 'user_idx', 'movie_idx', 'final_title', 'final_genres_str', 'poster_url', 'poster_url_source', 'poster_imdb_score', 'meta_vote_average', 'meta_vote_count', 'meta_popularity', 'meta_runtime', 'release_year', 'meta_original_language', 'meta_overview']


...

user2idx.csv
Size: 2.11 MB
Columns: ['userId', 'user_idx']


,userId,user_idx
0,1,0
1,2,1
2,3,2


audit_report.csv
Size: 0.00 MB
Columns: ['Unnamed: 0', 'value']


,Unnamed: 0,value
0,movies,62423
1,ratings_rows,25000095
2,users,162541


README.md
Size: 0.00 MB
# Multimodal Movie Dataset v2

Built by `dataset/merge.ipynb`, the unified dataset creation notebook.

Construction summary:
- MovieLens provides collaborative ratings and stable movie IDs.
- MovieLens links normalize IMDb and TMDB IDs for cross-source joins.
- MovieGenreFromItsPoster contributes legacy poster URLs, IMDb scores, and poster genres.
- TheMoviesDataset metadata is used when available.
- TMDB_all_movies is applied in the same build as fallback metadata and as the preferred poster so
